# 10 — Gestión del ciclo de vida de instancias

`IviumsoftInstanceManager` lanza, rastrea, adopta y cierra procesos de IviumSoft,
asignando cada proceso del sistema al **número de instancia del driver** con el que se
registra. Es la contraparte del *alcance* de instancias del notebook `02`: el alcance
decide *qué* instancia en ejecución recibe un comando; el gestor decide *qué instancias
existen*.

### Cuándo lo necesitas

- Arrancar N ventanas de IviumSoft de forma programática y controlarlas en paralelo
- Recuperarte tras un fallo: volver a asociar (`adopt`) instancias que sobrevivieron a tu script
- Limpiar procesos de IviumSoft sueltos dejados por una ejecución previa

### Notas

- **Solo Windows** — usa ayudantes nativos de proceso Win32.
- El driver debe estar abierto. En un arranque en frío **sin IviumSoft en ejecución
  todavía**, abre con `Pyvium.open_driver(verify_iviumsoft=False)` para que el gestor
  pueda lanzar la primera.
- La numeración de instancias del driver es estable tras un cierre: cerrar una deja un
  hueco en lugar de renumerar las supervivientes.

> Este notebook controla procesos reales, por lo que sus celdas no están pre-ejecutadas.
> Ejecútalo en una máquina Windows con IviumSoft instalado.

In [ ]:
from pyvium import Pyvium, IviumsoftInstanceManager
print("instance manager imported")

## 1. Arranque en frío

Abre el driver sin requerir un IviumSoft en ejecución, luego crea el gestor. Si tu
IviumSoft no está en la ruta por defecto `C:\\IviumStat\\IviumSoft.exe`, pasa `exe_path=...`.

In [ ]:
Pyvium.open_driver(verify_iviumsoft=False)

manager = IviumsoftInstanceManager(
    # exe_path=r"C:\IviumStat\IviumSoft.exe",   # cámbialo si está instalado en otra ruta
    launch_timeout=30.0,
    close_timeout=10.0,
)
print("driver open (cold start); manager ready")
print("already-active instances:", Pyvium.get_active_iviumsoft_instances())

## 2. Lanzar instancias

`launch()` arranca un proceso de IviumSoft y bloquea hasta que se registra con el driver,
luego devuelve su `ManagedInstance` (número de instancia + pid + hora de lanzamiento).
Serializa los lanzamientos internamente, así que el número de instancia se atribuye
correctamente aunque lances varias.

In [ ]:
first = manager.launch()
print("launched:", first)

second = manager.launch()
print("launched:", second)

print("instance numbers:", first.instance_number, second.instance_number)

## 3. Listar instancias

`list_instances()` devuelve un registro por cada instancia activa del driver. Las
gestionadas/adoptadas llevan un pid; las instancias que este gestor no abrió (huérfanas)
vuelven con `pid=None`. Los registros obsoletos (proceso desaparecido) se depuran.

In [ ]:
for record in manager.list_instances():
    kind = "managed" if record.managed else ("adopted/orphan")
    print(f"  instance {record.instance_number}: pid={record.pid} ({kind})")

## 4. Controlar una instancia gestionada

El gestor solo se ocupa del ciclo de vida; usa la API de alcance del notebook `02` para
enviar comandos. Aquí conectamos el dispositivo en la primera instancia lanzada y leemos
su estado.

In [ ]:
handle = Pyvium.instance(first.instance_number)

status, label = handle.get_device_status()
print(f"instance {first.instance_number}: status ({status}, '{label}')")

if status == 0:  # IviumSoft activo, dispositivo aún no conectado
    try:
        handle.connect_device()
        print("  connected, serial:", handle.get_device_serial_number())
    except Exception as error:
        print(f"  connect skipped: {type(error).__name__}: {error}")

## 5. Discover: reconciliar la vista del driver con la del SO

`discover()` es de solo lectura. Empareja lo que el gestor rastrea con lo que está
realmente en ejecución, separando las dos mitades que no puede emparejar automáticamente:
`orphan_instance_numbers` (lado del driver) y `untracked_processes` (lado del SO). En un
estado saludable los conteos coinciden.

In [ ]:
report = manager.discover()
print("tracked:")
for record in report.tracked:
    print(f"  instance {record.instance_number} (pid {record.pid})")
print("orphan instance numbers:", report.orphan_instance_numbers)
print("untracked processes:")
for process in report.untracked_processes:
    print(f"  pid {process.pid}  started {process.started_at}  title {process.window_title!r}")

## 6. Adoptar una instancia que el gestor no lanzó

Tras reiniciar un script, las ventanas de IviumSoft siguen en ejecución pero este gestor
no tiene registro de ellas. `adopt(instance_number, pid)` las vuelve a asociar. El pid
debe venir de una fuente externa que lo registrara, porque el driver no puede asignar
números de instancia a pids; `discover()` ayuda a emparejarlos por orden de lanzamiento
(el driver numera las instancias secuencialmente).

In [ ]:
# Flujo de recuperación ilustrativo: empareja cada proceso no rastreado (el más antiguo
# primero) con el número de instancia huérfana más bajo, luego adóptalo. Descomenta para
# ejecutarlo con huérfanas reales.
#
# report = manager.discover()
# for instance_number, process in zip(report.orphan_instance_numbers,
#                                     report.untracked_processes):
#     record = manager.adopt(instance_number, process.pid)
#     print("adopted:", record)
print("adopt() re-attaches an existing instance by (instance_number, pid)")

## 7. Cerrar una instancia

`close()` envía un cierre de ventana ordenado. Solo pueden cerrarse instancias con un pid
conocido (lanzadas o adoptadas).

Sus dos parámetros responden a dos preguntas distintas:

- **`on_measuring`** decide qué ocurre con una instancia que está midiendo, en el diálogo.
- **`force`** permite escalar a `TerminateProcess`, y nada más.

### Una instancia que mide responde con un diálogo

IviumSoft no se cierra mientras su dispositivo está midiendo: en su lugar levanta una
confirmación, una por cada ventana a la que el cierre envía el mensaje. `on_measuring`
elige el botón:

| `force` | `on_measuring` | resultado | número de instancia | medición |
|---|---|---|---|---|
| `False` | `'continue'` (por defecto) | cierre ordenado | liberado | **continúa en el dispositivo** |
| `False` | `'abort'` | cierre ordenado | liberado | termina |
| `False` | `'cancel'` | `DeviceBusyError`, no se envía nada | conservado | continúa |
| `False` | `None` | `TimeoutError`, el proceso sigue vivo | conservado | continúa |
| `True` | `'continue'` (por defecto) | cierre ordenado | liberado | **continúa en el dispositivo** |
| `True` | `'abort'` | cierre ordenado | liberado | termina |
| `True` | `'cancel'` | `DeviceBusyError`, sin matar nada | conservado | continúa |
| `True` | `None` | tiempo agotado y `TerminateProcess` | **se pierde** | continúa |

Con una instancia inactiva las ocho se comportan igual: se cierra y se libera el número.

El valor por defecto es el que interesa en hardware DataSecure: IviumSoft desaparece y el
experimento sigue corriendo. Usa `on_measuring='cancel'` para decir "nunca cierres una
instancia que está midiendo"; se comprueba antes de enviar nada y otra vez en el diálogo,
lo que cubre una instancia que empieza a medir entre medias.

### `force` solo autoriza la terminación

Un IviumSoft terminado nunca se da de baja, así que su número de instancia se pierde
mientras siga corriendo algún IviumSoft. Por eso requiere pedirlo explícitamente. Sin
`force`, un cierre que no llega a completarse lanza `TimeoutError` con el proceso todavía
vivo y el registro intacto, para que decidas tú:

```python
try:
    manager.close(n)
except TimeoutError:
    manager.terminate(n)      # explícito, y avisa de la pérdida del número
```

Una ventana que no sea la confirmación reconocida se informa con un aviso y se deja
intacta, porque pulsar Enter en un diálogo desconocido activaría el botón por defecto que
tenga, sea cual sea.

In [ ]:
manager.close(second.instance_number)
print("closed instance", second.instance_number)
print("remaining:", [r.instance_number for r in manager.list_instances()])

## 8. Barrer procesos huérfanos

`close_orphans()` cierra ordenadamente todos los procesos de IviumSoft que el gestor no
rastrea (los `untracked_processes` de `discover()`). Para instancias aún en uso, prefiere
`discover()` + `adopt()`.

Acepta los mismos `on_measuring` y `force` que `close()`, con el mismo significado, pero
nunca lanza una excepción por un solo proceso: un rechazo con `'cancel'` y, sin `force`,
un proceso que no llega a cerrarse se informan con un aviso y quedan fuera de los pids
devueltos. Que un huérfano se resista no es motivo para abandonar a los demás.

La excepción es `on_measuring='cancel'`, que es todo o nada: los pids huérfanos no pueden
emparejarse con números de instancia, así que si alguna instancia huérfana está midiendo,
no se cierra nada.

In [ ]:
# closed_pids = manager.close_orphans()          # añade force=True para forzar una ocupada
# print("closed orphan pids:", closed_pids)
print("close_orphans() sweeps untracked IviumSoft processes")

## Limpieza

Cierra lo que este notebook haya lanzado, luego cierra el driver.

In [ ]:
for record in manager.list_instances():
    if record.managed:
        try:
            manager.close(record.instance_number, force=True)
            print("closed", record.instance_number)
        except Exception as error:
            print(f"close {record.instance_number} failed: {type(error).__name__}: {error}")

Pyvium.close_driver()
print("driver closed")

---

## Resumen

| Tarea | API |
|------|-----|
| Apertura en frío (sin IviumSoft aún) | `Pyvium.open_driver(verify_iviumsoft=False)` |
| Crear el gestor | `IviumsoftInstanceManager(exe_path=..., ...)` |
| Lanzar una instancia | `.launch()` -> `ManagedInstance` |
| Listar instancias activas | `.list_instances()` |
| Reconciliar driver vs SO | `.discover()` -> `DiscoveryReport` |
| Volver a asociar tras reinicio | `.adopt(instance_number, pid)` |
| Cerrar una instancia | `.close(instance_number, on_measuring='continue')` |
| Matar una instancia atascada | `.terminate(instance_number)` (pierde el número) |
| Barrer procesos no rastreados | `.close_orphans(force=False)` |

## Siguiente

- **`02_device_and_instance_management.ipynb`** — delimitar comandos a una instancia / canal
- **`08_batch_and_synchronization.ipynb`** — coordinar mediciones entre instancias